# Laboratorio 2. El tablero no cuadra

Josué Alexander Rodríguez Alvarado 

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("ruta_csv", "/Volumes/workspace/lab2/archivos/ventas_export.csv")
ruta = dbutils.widgets.get("ruta_csv")

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# 1) Normalizar monto: "12,50" -> 12.50. "1.250,00" (miles con punto) NO se adivina.
spark.sql(r"""
create or replace temp view base as
select *,
  case
    when trim(monto) rlike '^-?[0-9]+([.][0-9]+)?$' then trim(monto)
    when trim(monto) rlike '^-?[0-9]+,[0-9]+$' then replace(trim(monto), ',', '.')
  end as monto_norm
from bronce_ventas
""")

# 2) Motivos que se detectan fila por fila
spark.sql("""
create or replace temp view base2 as
select *,
  try_cast(monto_norm as decimal(12,2)) as monto_num,
  concat_ws('|', fecha, tienda_id, producto_id, cantidad, monto_norm, medio_pago) as firma,
  case
    when try_cast(trim(tienda_id) as int) is null
      or try_cast(trim(tienda_id) as int) not in (1, 2, 3) then 'tienda fuera de alcance (solo 1, 2 y 3)'
    when trim(monto) rlike '^-?[0-9]{1,3}([.][0-9]{3})+,[0-9]+$' then 'monto con formato ambiguo (miles con punto y decimales con coma)'
    when try_cast(monto_norm as decimal(12,2)) is null then 'monto no numerico o vacio'
    when try_cast(monto_norm as decimal(12,2)) < 0 then 'monto negativo'
  end as motivo_previo
from base
""")

# 3) Entre las filas válidas: ¿el mismo venta_id trae datos distintos? ¿es copia exacta?
spark.sql("""
create or replace temp view validas as
select *,
  size(collect_set(firma) over (partition by venta_id)) as versiones,
  row_number() over (partition by venta_id order by firma) as rn
from base2
where motivo_previo is null
""")

# 4) Hecho: sin la columna canal
spark.sql("""
create or replace temp view oro_fact_ventas as
select venta_id, to_date(fecha) as fecha, cast(tienda_id as int) as tienda_id,
       cast(producto_id as int) as producto_id, cast(cantidad as int) as cantidad,
       monto_num as monto, medio_pago
from validas
where versiones = 1 and rn = 1
""")

# 5) Cuarentena con motivo
spark.sql("""
create or replace temp view cuarentena_ventas as
select venta_id, fecha, tienda_id, producto_id, cantidad, monto, medio_pago, canal,
       motivo_previo as motivo
from base2 where motivo_previo is not null
union all
select venta_id, fecha, tienda_id, producto_id, cantidad, monto, medio_pago, canal,
       'venta_id repetida con datos distintos (conflicto)' as motivo
from validas where versiones > 1
union all
select venta_id, fecha, tienda_id, producto_id, cantidad, monto, medio_pago, canal,
       'duplicado exacto (se conserva una copia en oro)' as motivo
from validas where versiones = 1 and rn > 1
""")

display(spark.sql("select motivo, count(*) as filas from cuarentena_ventas group by motivo order by filas desc"))

## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

In [0]:
display(spark.sql("select count(*) as filas_oro from oro_fact_ventas"))

## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
2. ¿Qué hicieron con un monto que venía en texto?
3. ¿Por qué una tienda se quedó fuera?
4. ¿Dónde quedó la columna que no estaba en el contrato?
5. Si corren el notebook otra vez, ¿el hecho crece?

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio

**Respuestas al Desafío
**
1. ¿Qué pasó con la misma venta llegada dos veces?  
Usé ROW_NUMBER() OVER (PARTITION BY venta_id ...) para conservar la primera fila y mandar duplicados a cuarentena. Si traía datos distintos, como la venta 3050 (montos 10.00 y 10.50), aparté ambas filas por inconsistencia tras comparar la firma de la celda.

2. ¿Qué hicieron con un monto que venía en texto?  
Reemplacé la coma por punto y los convertí a DECIMAL(12,2) para evitar errores de redondeo. Aparté 4 filas vacías/"N/D"/"pendiente", 1 de formato ambiguo ("1.250,00") y 1 con monto negativo (3114) por criterio de prudencia.

3. ¿Por qué una tienda se quedó fuera?  
El alcance exigía solo las tiendas 1, 2 y 3. Aparté 22 filas de las tiendas 4, 5, 6, 7, 77, 88 y 99. Aunque la 5 y la 6 existen en la base maestra, quedaron fuera por no estar en el contrato.

4. ¿Dónde quedó la columna que no estaba en el contrato?   
La columna canal no pertenecía al contrato de oro_fact_ventas, así que no la incluí en el hecho final, pero la mantuve en cuarentena_ventas para no perder la trazabilidad de origen.

5. Si corren el notebook otra vez, ¿el hecho crece?   
No. Todo se calcula sobre CREATE OR REPLACE TEMP VIEW desde bronce. Lo probé reejecutando el notebook y obtuve exactamente las mismas 24 filas en oro_fact_ventas. (En producción con Delta se usaría MERGE INTO).